In [ ]:
import numpy
import torch.nn as nn
import torch

#input parameters
n=336 #batch size
l=50 #sequence len
c=4 #embedding dimension (called it f for the LSTM example)

#model parameters
k = 3 #kernel size
s = 4 #stride
p = 2 #pool size

m = nn.AvgPool1d(p)


In [46]:
#Generate input [336, 50, 4]
rand_arr = torch.rand(n,l,c) 
print(rand_arr.shape)

torch.Size([336, 50, 4])


In [47]:
#Permute into correct shape (batch, embeddings, sequence len)
rand_arr_permute = rand_arr.permute(0,2,1)
print(rand_arr_permute.shape)

torch.Size([336, 4, 50])


In [48]:
conv1 = nn.Conv1d(out_channels=32, in_channels=c, kernel_size=k)
out1 = conv1(rand_arr_permute)
print(out1.shape)
out1= m(out1)
print(out1.shape)

torch.Size([336, 32, 48])
torch.Size([336, 32, 24])


In [49]:
conv2 = nn.Conv1d(out_channels=16, in_channels=32, kernel_size=k)
out2 = conv2(out1)
print(out2.shape)
out2= m(out2)
print(out2.shape)

torch.Size([336, 16, 22])
torch.Size([336, 16, 11])


In [50]:
conv3 = nn.ConvTranspose1d(out_channels=32, in_channels=16, kernel_size=k, stride=s)
out3 = conv3(out2)
print(out3.shape)
out3= m(out3)
print(out3.shape)

torch.Size([336, 32, 43])
torch.Size([336, 32, 21])


In [51]:
conv4 = nn.ConvTranspose1d(out_channels=c, in_channels=32, kernel_size=k, stride=s)
out4 = conv4(out3)
print(out4.shape)
out4= m(out4)
print(out4.shape)

torch.Size([336, 4, 83])
torch.Size([336, 4, 41])


In [52]:
dense = nn.Linear(out4.shape[-1], l)
out5 = dense(out4)
print(out5.shape)


torch.Size([336, 4, 50])


In [43]:
#size remains same; just some values are zeroed out
m = nn.Dropout(p=0.2)
input_ = torch.randn(20, 16)
output = m(input_) 

In [44]:
# pool with window of size=2, stride=2 (default is same as) -> halves the sequence length
m = nn.AvgPool1d(2)
print (input_.shape)
m(input_).shape

torch.Size([20, 16])


torch.Size([20, 8])

In [ ]:
class ConvAutoEncoder(nn.Module):
    def __init__(self, seq_len):
        super(ConvAutoEncoder, self).__init__()
        self.seq_len = 5 #this is actually the number of features we have, but xoz of editing LSTM code, it's named as it was before
        self.kernel_size = 7
        self.pool_size = 2

        self.encode = nn.Sequential(
            nn.Conv1d(out_channels=32, in_channels=self.seq_len, kernel_size=self.kernel_size),
            nn.Tanh(),
            nn.AvgPool1d(self.pool_size),
            nn.Dropout(0.2),
            nn.Conv1d(out_channels=16, in_channels=32, kernel_size=self.kernel_size),
            nn.AvgPool1d(self.pool_size),
            nn.Dropout(0.2)
        )

        self.decode = nn.Sequential(
            nn.ConvTranspose1d(out_channels=32, in_channels=16, kernel_size=self.kernel_size, stride=4),
            nn.Tanh(),
            nn.AvgPool1d(self.pool_size),
            nn.Dropout(0.2),
            nn.ConvTranspose1d(out_channels=self.seq_len, in_channels=32, kernel_size=self.kernel_size, stride=4),
            nn.AvgPool1d(self.pool_size),
            nn.Dropout(0.2),
            nn.Linear(271, 288)
        )

    def forward(self, x):
        x = self.encode(x)
        x = self.decode(x)
        return x